# Tanager Mangrove Mapping - 01b Multispectral-Equivalent Preprocessing

| | |
|---|---|
| **Authors**     | Muhammad Wahyu Ramadhan, Athar Abdurrahman B., Diniyarti |
| **Competition** | Planet Tanager Open Data Competition 2026 |
| **Topic**       | Transferable Mangrove Extent and Biomass Mapping Using Adaptive Spectral Thresholds |
| **Date**        | July 2026 |

---

**Scope (Scenario B, journal-only):** Load Tanager HDF5, convolve with Sentinel-2 SRF to produce multispectral-equivalent bands (B3, B4, B8, B11), compute reduced feature stack (NDMI, MVI, MNDWI, SAVI), and generate pseudo-labels for downstream classification. REIP and EMI are excluded by design (not computable from S2 bandpasses).

**Note:** This notebook mirrors `01_preprocessing.ipynb` but replaces the discrete-wavelength band extraction with SRF resampling. Downstream classification (`02b`) and transferability (`04b`) will consume outputs from this notebook.

## 0. Environment Setup

In [ ]:
# !pip install h5py scikit-image geopandas rasterio matplotlib openpyxl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import h5py

# ============================================================
# Project root and Scenario B paths
# ============================================================
ROOT           = Path('/content/drive/MyDrive/PROJECT/Planet Tanager Competition 2026/tanager-mangrove-mapping')
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROC      = ROOT / 'data' / 'processed'         # Scenario A outputs (reference)
DATA_PROC_S2EQ = ROOT / 'data' / 'processed_s2eq'    # Scenario B outputs (this notebook)
DATA_SRF       = ROOT / 'data' / 'sentinel2_srf'
OUT_FIGURES    = ROOT / 'outputs' / 'figures'

for d in (DATA_PROC_S2EQ, OUT_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))

# ============================================================
# Reload src modules during development
# ============================================================
import src.preprocessing        as _pre
import src.spectral_resampling  as _srf
importlib.reload(_pre)
importlib.reload(_srf)

from src.spectral_resampling import (
    S2_TARGET_BANDS,
    load_sentinel2_srf,
    resample_hyperspectral_to_s2,
    compute_scenario_b_indices,
    write_scenario_b_outputs,
)

# Site registry (identical to Scenario A)
SITES = {
    'sangatta'   : '20250302_030003_92_4001',
    'gujarat'    : '20250311_061550_53_4001',
    'elsalvador' : '20250223_165546_32_4001',
    'belize'     : '20250824_171857_84_4001',
    'australia'  : '20250608_014315_58_4001',
}

print(f'ROOT           : {ROOT}')
print(f'Sites          : {list(SITES.keys())}')
print(f'Target S2 bands: {list(S2_TARGET_BANDS.keys())}')


## 1. Load Sentinel-2 Spectral Response Function

Source: ESA COPE-GSEG-EOPG-TN-15-0007. See `data/sentinel2_srf/README.md` for download instructions. Default platform: Sentinel-2A.

In [ ]:
# ============================================================
# Load SRF once, reuse across all sites
# ============================================================
SRF_FILE = DATA_SRF / 's2_srf_v5.0.xlsx'   # rename to match your download

srf_dict = load_sentinel2_srf(SRF_FILE, platform='S2A')

for band_code, (wl, w) in srf_dict.items():
    print(f'  {band_code:<4} : {len(wl):>4} points, {wl.min():.0f}-{wl.max():.0f} nm')


## 2. Per-Site Resampling Loop

For each site:
1. Load HDF5 (426 bands)
2. Convolve with S2 SRF -> 4 equivalent bands
3. Compute NDMI, MVI, MNDWI, SAVI
4. Write outputs to `data/processed_s2eq/`

Reference GeoTIFF (`bands.tif` from Scenario A) is used to inherit CRS and transform, ensuring pixel-perfect alignment between scenarios.

In [ ]:
# ============================================================
# Resample each site and save Scenario B outputs
# ============================================================
for site, scene_id in SITES.items():
    print(f'\n{"="*60}')
    print(f'  Site : {site}  ({scene_id})')
    print(f'{"="*60}')

    h5_path       = DATA_RAW  / f'{site}_{scene_id}_ortho_sr_hdf5.h5'
    reference_tif = DATA_PROC / f'{site}_{scene_id}_bands.tif'
    site_key      = f'{site}_{scene_id}'

    # ---- Load HDF5 spectrum (TODO: reuse _pre.load_hdf5 if signature matches)
    with h5py.File(h5_path, 'r') as f:
        # TODO: extract reflectance cube (n_bands, H, W) and wavelengths
        tanager_stack       = None
        tanager_wavelengths = None

    # ---- Resample to S2-equivalent bands
    resampled = resample_hyperspectral_to_s2(
        tanager_stack, tanager_wavelengths, srf_dict
    )

    # ---- Compute Scenario B indices
    indices_b = compute_scenario_b_indices(resampled)

    # ---- Write outputs
    write_scenario_b_outputs(
        indices_b,
        reference_tif = reference_tif,
        out_dir       = DATA_PROC_S2EQ,
        site_key      = site_key,
    )
    print(f'  Outputs saved  : {DATA_PROC_S2EQ}')


## 3. Sanity Check: Scenario A vs Scenario B Reflectance

Quick visual check that resampled S2-equivalent bands are consistent with the discrete-wavelength Scenario A bands. Not a validation step, just to confirm resampling did not corrupt spectral magnitude order.

Expected: S2-equivalent reflectance close but not identical to Scenario A discrete bands (SRF averages over ~30-100 nm windows, discrete extraction uses single-band point sampling).

In [ ]:
# ============================================================
# Compare mean reflectance per band: Scenario A vs B (Sangatta)
# ============================================================
# TODO: implement after resample function is working


## 4. Adaptive Threshold and Pseudo-Label Generation (Scenario B)

Reuses `apply_adaptive_threshold` and `compute_coastal_candidate_mask` from `src/preprocessing.py`, applied to the Scenario B feature stack.

This ensures the only difference between scenarios is the feature stack itself. Pseudo-label generation logic is held constant.

In [ ]:
# ============================================================
# Adaptive threshold per site using Scenario B indices
# ============================================================
# TODO: implement after resample function is working
#       reuse: _pre.compute_coastal_candidate_mask
#              _pre.apply_adaptive_threshold (force_otsu on MVI)
